In [ ]:
from google.colab import files
uploaded = files.upload()

TypeError: 'NoneType' object is not subscriptable

In [ ]:
with open("transcript.txt", "r", encoding="utf-8") as f:
    transcript = f.read()

print(transcript[:500])   # Preview first 500 characters

In [ ]:
!pip install nltk transformers torch

In [ ]:
import nltk
import re
from nltk.tokenize import sent_tokenize
from transformers import BertTokenizer

In [ ]:
nltk.download('punkt')

In [ ]:
with open("transcript.txt", "r", encoding="utf-8") as f:
    transcript = f.read()

print(transcript[:500])   # preview

In [ ]:
lines = transcript.strip().split("\n")

In [ ]:
sentence_data = []

for line in lines:
    match = re.match(r"(\d+:\d+)\s+(.*)", line)

    if match:
        timestamp = match.group(1)
        text = match.group(2)

        # Divide into sentences using NLTK
        sentences = sent_tokenize(text)

        for sentence in sentences:
            sentence_data.append({
                "timestamp": timestamp,
                "sentence": sentence
            })

print(sentence_data[:5])

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

In [ ]:
for item in sentence_data:
    tokens = tokenizer.tokenize(item["sentence"])
    item["bert_tokens"] = tokens

In [ ]:
for item in sentence_data[:3]:
    print("Timestamp:", item["timestamp"])
    print("Sentence:", item["sentence"])
    print("BERT Tokens:", item["bert_tokens"])
    print("-" * 50)

In [ ]:
import re

def clean_text(text):
    text = re.sub(r"\d+:\d+", "", text)   # remove timestamps like 0:23
    text = re.sub(r"\n", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

transcript = clean_text(transcript)

In [ ]:
def chunk_text(text, chunk_size=500):
    chunks = []
    for i in range(0, len(text), chunk_size):
        chunks.append(text[i:i+chunk_size])
    return chunks

chunks = chunk_text(transcript)

print("Total chunks:", len(chunks))
print(chunks[0])

Total chunks: 15
LangChain and LangGraph are both open source frameworks designed to help developers build applications with large language models. So what are the differences and why use one over the other? Well, I think a good place to start. Is to define what these two things are, and let's begin with LangChain. LangChain Now, we've done a dedicated video on LangChain, and my guess is there's probably a pop up somewhere over my head right now encouraging you to watch that video. But don't click, not yet. Give


In [ ]:
import re

def fix_broken_words(text):
    # Join words broken by newline
    text = re.sub(r'\n', ' ', text)

    # Fix word breaks like "m ponents" or "en finally"
    text = re.sub(r'(\w)\s+(\w)', r'\1 \2', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    return text.strip()

cleaned_text = fix_broken_words(transcript)

In [ ]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab') # Added to resolve LookupError

from nltk.tokenize import sent_tokenize

sentences = sent_tokenize(cleaned_text)

for i, s in enumerate(sentences[:10]):
    print(f"Sentence {i+1}: {s}")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Sentence 1: LangChain and LangGraph are both open source frameworks designed to help developers build applications with large language models.
Sentence 2: So what are the differences and why use one over the other?
Sentence 3: Well, I think a good place to start.
Sentence 4: Is to define what these two things are, and let's begin with LangChain.
Sentence 5: LangChain Now, we've done a dedicated video on LangChain, and my guess is there's probably a pop up somewhere over my head right now encouraging you to watch that video.
Sentence 6: But don't click, not yet.
Sentence 7: Give me a moment to summarize LangChain.
Sentence 8: Then at the end of this video, if you want any more, you can go back and check that one out.
Sentence 9: Okay.
Sentence 10: Now, at its core, LangChain is a way for building LLM powered applications by executing a sequence of functions in a chain.


In [ ]:
def chunk_by_sentences(sentences, chunk_size=3):
    chunks = []
    for i in range(0, len(sentences), chunk_size):
        chunk = " ".join(sentences[i:i+chunk_size])
        chunks.append(chunk)
    return chunks

chunks = chunk_by_sentences(sentences, chunk_size=3)

In [ ]:
for i, chunk in enumerate(chunks):
    print(f"\nChunk {i+1}")
    print("-" * 40)
    print(chunk)


Chunk 1
----------------------------------------
LangChain and LangGraph are both open source frameworks designed to help developers build applications with large language models. So what are the differences and why use one over the other? Well, I think a good place to start.

Chunk 2
----------------------------------------
Is to define what these two things are, and let's begin with LangChain. LangChain Now, we've done a dedicated video on LangChain, and my guess is there's probably a pop up somewhere over my head right now encouraging you to watch that video. But don't click, not yet.

Chunk 3
----------------------------------------
Give me a moment to summarize LangChain. Then at the end of this video, if you want any more, you can go back and check that one out. Okay.

Chunk 4
----------------------------------------
Now, at its core, LangChain is a way for building LLM powered applications by executing a sequence of functions in a chain. So let's say we're building an applicati

In [ ]:
import os
os.environ["OPENAI_API_KEY"] =

In [ ]:
from openai import OpenAI
client = OpenAI()


In [ ]:
from openai import OpenAI
client = OpenAI()

def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

embeddings = [get_embedding(chunk) for chunk in chunks]

In [ ]:
pip install psycopg2-binary

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 42.3 MB/s eta 0:00:00


In [ ]:
!pip install faiss-cpu

In [ ]:
import faiss
import numpy as np

# Convert embeddings to numpy
vectors = np.array(embeddings).astype("float32")

# Get dimension
dimension = vectors.shape[1]

# Create FAISS index
index = faiss.IndexFlatL2(dimension)

# Add vectors
index.add(vectors)

print("Total vectors stored:", index.ntotal)

Total vectors stored: 25


In [ ]:
def retrieve(query, k=3):
    query_embedding = get_embedding(query)
    query_vector = np.array([query_embedding]).astype("float32")

    distances, indices = index.search(query_vector, k)

    results = [chunks[i] for i in indices[0]]
    return results

In [ ]:
def ask_rag(query):
    retrieved_chunks = retrieve(query)

    context = "\n\n".join(retrieved_chunks)

    prompt = f"""
    Answer only from the context below.
    If answer not found, say: Question is irrelevant.

    Context:
    {context}

    Question:
    {query}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

In [ ]:
print(ask_rag("What is the difference between LangChain and LangGraph?"))

LangChain is a modular architecture that allows for the building of complex workflows by combining high-level components, while LangGraph is a specialized library within the LangChain ecosystem focused specifically on creating and managing stateful multi-agent systems and handling complex nonlinear workflows.


In [ ]:
print(ask_rag ("What is LangChain and what problem does it solve?"))

LangChain is an abstraction layer for chaining LLM operations into large language model applications. It provides a way to build LLM powered applications by executing a sequence of functions in a chain, solving the problem of organizing and coordinating tasks needed in such applications, like retrieving data and summarizing it.


In [ ]:
print(ask_rag("What does DAG stand for in LangChain architecture?"))

DAG stands for directed acyclic graph.
